# Gold: hospital_daily_summary

Builds `azuredb_caresync.gold.hospital_daily_summary` (one row per hospital per day) from the silver layer per the STTM `sttm_silver_to_gold.xlsx`.

* **Driving table:** `silver.lab_results` grouped by `hospital_id`, `test_date`
* **Joins:** LEFT JOIN `silver.hospitals`; LEFT JOIN `silver.calendar` on `date_key`
* **Window:** `LAG(total_tests)` per hospital ordered by `summary_date` for day-over-day trend
* **Load type:** FULL rebuild — `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS azuredb_caresync.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE azuredb_caresync.gold.hospital_daily_summary
COMMENT 'Gold: one row per hospital per day with lab test volume, abnormal/critical rates and day-over-day trend'
AS
WITH daily AS (
  -- Lab results aggregated to hospital x day grain
  SELECT
    hospital_id,
    test_date                              AS summary_date,
    COUNT(*)                               AS total_tests,
    COUNT(DISTINCT patient_id)             AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)              AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')   AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')   AS critical_test_count
  FROM azuredb_caresync.silver.lab_results
  GROUP BY hospital_id, test_date
),
lagged AS (
  SELECT
    *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)
SELECT
  d.hospital_id,
  h.hospital_name,
  d.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(d.total_tests AS INT)                                              AS total_tests,
  CAST(d.unique_patients_tested AS INT)                                   AS unique_patients_tested,
  CAST(d.active_doctor_count AS INT)                                      AS active_doctor_count,
  CAST(d.abnormal_test_count AS INT)                                      AS abnormal_test_count,
  CAST(d.critical_test_count AS INT)                                      AS critical_test_count,
  CAST(d.abnormal_test_count * 100.0 / d.total_tests AS DECIMAL(5,2))     AS abnormal_rate_pct,
  CAST(d.critical_test_count * 100.0 / d.total_tests AS DECIMAL(5,2))     AS critical_rate_pct,
  CAST(d.prev_day_total_tests AS INT)                                     AS prev_day_total_tests,
  CAST(
    CASE WHEN d.prev_day_total_tests IS NULL OR d.prev_day_total_tests = 0 THEN NULL
         ELSE (d.total_tests - d.prev_day_total_tests) * 100.0 / d.prev_day_total_tests
    END AS DECIMAL(6,2))                                                  AS test_volume_change_pct
FROM lagged d
LEFT JOIN azuredb_caresync.silver.hospitals h
  ON d.hospital_id = h.hospital_id
LEFT JOIN azuredb_caresync.silver.calendar c
  ON c.date_key = CAST(date_format(d.summary_date, 'yyyyMMdd') AS INT);

In [0]:
updated_df = spark.table("azuredb_caresync.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))